| **Retriever Type**                   | **Purpose**                                                                         | **Ideal Usage Scenario**                                                                                                       |
| ------------------------------------ | ----------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------ |
| **VectorStoreRetriever**             | Retrieve documents based on vector similarity search (e.g. FAISS, Chroma, Pinecone) | Basic RAG (Retrieval-Augmented Generation) setup for fetching semantically relevant documents                                  |
| **ContextualCompressionRetriever**   | Compresses retrieved docs using an LLM to return only the most relevant parts       | When using large context docs (PDFs, transcripts) where only a portion of each doc is relevant                                 |
| **MultiQueryRetriever**              | Generates multiple rephrased queries to improve retrieval coverage                  | When a user query can be ambiguous or phrased differently; improves diversity of retrieved documents                           |
| **ParentDocumentRetriever**          | Retrieves a full parent document instead of the split chunk                         | When a chunk is matched but full document context is required (e.g., blog, article, contract analysis)                         |
| **BM25Retriever**                    | Keyword-based retriever using classic BM25 algorithm                                | When you want to retrieve based on exact keyword matches (useful for legal, medical, or short documents with precise language) |
| **EnsembleRetriever**                | Combines multiple retrievers (e.g., vector + BM25) with weighted scoring            | When you want to merge semantic and keyword-based search for better hybrid performance                                         |
| **TimeWeightedVectorStoreRetriever** | VectorStoreRetriever + time-based decay to prefer recent documents                  | Best for chatbot memory or news search where **recency** of information is important                                           |
| **TavilySearchAPIRetriever**         | Uses external web search engine via Tavily API to retrieve **live web data**        | When querying real-world events, breaking news, or updated information outside your local docs or vector database              |


In [1]:
#1)VectorStoreRetriever	Vector-based	Embedding similarity search	General-purpose RAG
# ConversationalRetrievalChain
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import Tool
from langchain.vectorstores import FAISS
from langchain.chains import ConversationalRetrievalChain
from langchain.agents import initialize_agent, AgentType
from langchain.tools import StructuredTool
from langchain.memory import ConversationBufferMemory
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


# 1. Setup
GROQ_API_KEY = ""

# Initialize LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0,
    max_tokens=300,
)

# 2. Vector DB
with open("sample.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

# 🧠 Split the text into smaller chunks
from langchain.text_splitter import CharacterTextSplitter
splitter = CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
texts = splitter.split_text(text_data)

embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_texts(texts, embedding)
retriever = vectorstore.as_retriever()

# 3. Conversational RAG chain
rag_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    return_source_documents=False
)

# 4. Memory for chat history
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# 5. Wrap RAG as a StructuredTool
def rag_tool_fn(question: str) -> str:
    return rag_chain.invoke({
        "question": question,
        "chat_history": []
    })["answer"]

rag_tool = StructuredTool.from_function(
    name="RAG_QA",
    description="Use this to answer questions about LangChain.",
    func=rag_tool_fn
)

# 6. Create agent
agent = initialize_agent(
    tools=[rag_tool],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True,
    memory=memory,
    handle_parsing_errors=True
)

# 7. Run conversation
print("1️⃣ First question")
res1 = agent.run("What is LangChain?")
print("Answer:", res1)

print("\n2️⃣ Follow-up")
res2 = agent.run("Who created it?")
print("Answer:", res2)

print("\n3️⃣ Ask again")
res3 = agent.run("Explain LangChain again simply.")
print("Answer:", res3)


C:\Users\PAVAN\AppData\Local\Temp\ipykernel_17740\2665710180.py:36: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
C:\Users\PAVAN\anaconda3\envs\lang_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 1690.89it/s]
C:\Users\PAVAN\AppData\Local\Temp\ipykernel_17740\2665710180.py:64: LangChainDeprecationWarning: The function `initialize_agent`

1️⃣ First question
Thought: Do I need to use a tool? Yes
Action: RAG_QA
Action Input: What is LangChain?
Observation: LangChain is a framework for building applications with Large Language Models (LLMs). It was created by Harrison Chase and supports various features such as RAG, agents, memory, tools, and more. It's commonly used in applications like chatbots, document Q&A, and AI workflows.
Thought:

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Do I need to use a tool? No
AI: LangChain is a framework for building applications with Large Language Models (LLMs). It was created by Harrison Chase and supports various features such as RAG, agents, memory, tools, and more. It's commonly used in applications like chatbots, document Q&A, and AI workflows.

> Finished chain.
Answer: LangChain is a framework for building applications with Large Language Models (LLMs). It was created by Harrison Chase and supports various features such as RAG, agents, memory, tools, and more. It's commonly used in applications like chatbots, document Q&A, and AI workflows.

2️⃣ Follow-up
Thought: Do I need to use a tool? Yes
Action: RAG_QA
Action Input: Who created LangChain?
Observation: Harrison Chase created LangChain.
Thought:

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Thought: Do I need to use a tool? No
AI: Harrison Chase created LangChain.

> Finished chain.
Answer: Harrison Chase created LangChain.

3️⃣ Ask again
Thought: Do I need to use a tool? No
AI: LangChain is a framework for building applications with Large Language Models (LLMs). It was created by Harrison Chase and supports various features such as RAG, agents, memory, tools, and more.

> Finished chain.
Answer: LangChain is a framework for building applications with Large Language Models (LLMs). It was created by Harrison Chase and supports various features such as RAG, agents, memory, tools, and more.
